In [1]:
import pandas as pd


In [2]:
bioinfo = pd.read_parquet('../data/raw/bioinfo.parquet')
ideology = pd.read_parquet('../data/raw/ideology.parquet')
terms = pd.read_parquet('../data/raw/terms.parquet')

committee_members = pd.read_parquet('../data/raw/committee_members.parquet')
committees = pd.read_parquet('../data/raw/committees.parquet')

bill_summaries_119 = pd.read_parquet('../data/raw/bill_summaries_119.parquet')
schedule_e_2026 = pd.read_parquet('../data/raw/schedule_e_2026.parquet')
sponsored_legislation = pd.read_parquet('../data/raw/sponsored_legislation.parquet')

bioinfo.info()


<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bioguide_id     555 non-null    str    
 1   Full name       555 non-null    str    
 2   Chamber         555 non-null    str    
 3   State           555 non-null    str    
 4   Party           555 non-null    str    
 5   District        439 non-null    float64
 6   birthYear       554 non-null    str    
 7   image           554 non-null    str    
 8   Office address  539 non-null    str    
 9   Phone           539 non-null    str    
 10  Website         539 non-null    str    
dtypes: float64(1), str(10)
memory usage: 166.9 KB


In [3]:
bioinfo['bioguide_id'].duplicated().value_counts()

bioguide_id
False    555
Name: count, dtype: int64

In [4]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
Full name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
Chamber,House of Representatives,House of Representatives,House of Representatives
State,Alabama,Alabama,Alabama
Party,Republican,Democratic,Republican
District,3.0,7.0,6.0
birthYear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
Office address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
Phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [12]:
ideology = ideology[['icpsr', 'bioguide_id', 'fec_id', 'left_right_ideology']].groupby('icpsr')['left_right_ideology'].mean()

In [13]:
ideology

icpsr
14226    0.363
14435   -0.518
14854    0.334
14858   -0.356
14863    0.184
         ...  
49308   -0.357
49703    0.124
91980    0.384
92336    0.152
99912    0.403
Name: left_right_ideology, Length: 557, dtype: float64

In [17]:
terms = terms.drop(['startYear', 'endYear'], axis = 1)

In [25]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.

    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.

    Returns:
        A DBML string representing the DataFrame schema.
    """

    dbml_string = f"Table {table_name} {{\n"

    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"

    dbml_string += "}\n"
    return dbml_string

def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [ ]:
print(pandas_df_to_dbml(bioinfo, 'members'))

Table members {
  bioguide_id varchar
  Full name varchar
  Chamber varchar
  State varchar
  Party varchar
  District float
  birthYear varchar
  image varchar
  Office address varchar
  Phone varchar
  Website varchar
}



In [ ]:
members = pd.merge(bioinfo, ideology,
                   on = 'bioguide_id',
                   how ='outer',
                   indicator = 'matched')

In [ ]:
members

In [3]:
bill_summaries_119.head(3).T

,0,1,2
actionDate,2025-01-03,2025-01-09,2025-01-06
actionDesc,Introduced in House,Introduced in House,Introduced in Senate
currentChamber,House,House,Senate
currentChamberCode,H,H,S
lastSummaryUpdateDate,2025-01-09T01:58:40Z,2025-01-13T12:49:11Z,2025-01-13T16:21:37Z
text,<p><strong>Illegitimate Court Counteraction Ac...,<p>This resolution disapproves of the Central ...,<p><strong>Laken Riley Act</strong></p><p>This...
updateDate,2025-01-09T01:58:53Z,2025-01-13T12:51:00Z,2025-01-13T16:21:43Z
versionCode,00,00,00
bill.congress,119,119,119
bill.number,23,27,5


In [ ]:
from bs4 import BeautifulSoup

mysoup = BeautifulSoup(bill_summaries_119['text'][0], 'html.parser')

<p><strong>Illegitimate Court Counteraction Act</strong></p><p>This bill imposes sanctions against foreign persons (individuals and entities) who assist the International Criminal Court (ICC) in investigating, arresting, detaining, or prosecuting certain individuals.</p><p>The bill categorizes as protected persons (1) any U.S. individual, U.S. entity, or person in the United States, unless the United States is a state party to the Rome Statute of the ICC and provides formal consent to ICC jurisdiction; and (2) any foreign person that is a citizen or lawful resident of a U.S. ally that is not a state party to the Rome Statute or has not consented to ICC jurisdiction.</p><p>If the ICC attempts to investigate, arrest, detain or prosecute a protected person, the President must impose visa- and property-blocking sanctions against the foreign persons that engaged in or materially assisted in such actions, as well as against foreign persons owned by, controlled by, or acting on behalf of such

In [5]:
sponsored_legislation.head(3).T

,0,1,2
introducedDate,2026-05-13,2026-07-22,2026-07-22
number,8800,NaN,NaN
title,National Defense Authorization Act for Fiscal ...,NaN,NaN
type,HR,NaN,NaN
url,https://api.congress.gov/v3/bill/119/hr/8800?f...,https://api.congress.gov/v3/amendment/119/hamd...,https://api.congress.gov/v3/amendment/119/hamd...
bioguide_id,R000575,R000575,R000575


In [10]:
sponsored_legislation[sponsored_legislation['url'].duplicated(keep=False)]['url']

13937    https://api.congress.gov/v3/amendment/119/samd...
13938    https://api.congress.gov/v3/amendment/119/samd...
15083    https://api.congress.gov/v3/bill/119/s/3872?fo...
15085    https://api.congress.gov/v3/bill/119/s/3872?fo...
20372    https://api.congress.gov/v3/amendment/119/samd...
20373    https://api.congress.gov/v3/amendment/119/samd...
20374    https://api.congress.gov/v3/amendment/119/samd...
20375    https://api.congress.gov/v3/amendment/119/samd...
20379    https://api.congress.gov/v3/amendment/119/samd...
20380    https://api.congress.gov/v3/amendment/119/samd...
20381    https://api.congress.gov/v3/amendment/119/samd...
20382    https://api.congress.gov/v3/amendment/119/samd...
21793    https://api.congress.gov/v3/amendment/119/samd...
21796    https://api.congress.gov/v3/amendment/119/samd...
25090    https://api.congress.gov/v3/amendment/119/samd...
25091    https://api.congress.gov/v3/amendment/119/samd...
25097    https://api.congress.gov/v3/amendment/119/samd.

In [18]:
sponsored_legislation = sponsored_legislation.drop_duplicates()
sponsored_legislation = sponsored_legislation[['bioguide_id', 'url']]

bills = pd.merge(bill_summaries_119, sponsored_legislation,
                 left_on=['bill.url'], right_on=['url'],
                 how = 'outer',
                 indicator='matched',
                 validate = 'many_to_one')

bills = bills.query("matched == 'both'")
bills = bills.drop('matched', axis=1)
bills.columns = [x.lower().replace('.', '_') for x in bills.columns]

bills

,actiondate,actiondesc,currentchamber,currentchambercode,lastsummaryupdatedate,text,updatedate,versioncode,bill_congress,bill_number,bill_originchamber,bill_originchambercode,bill_title,bill_type,bill_updatedateincludingtext,bill_url,bioguide_id,url
7021,2026-05-13,Introduced in House,House,H,2026-06-24T14:07:00Z,<p>This concurrent resolution directs the Pres...,2026-06-24T14:07:20Z,00,119.0,100,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,F000477,https://api.congress.gov/v3/bill/119/hconres/1...
7022,2026-05-14,Introduced in House,House,H,2026-07-18T16:28:55Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:00Z,00,119.0,101,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,J000305,https://api.congress.gov/v3/bill/119/hconres/1...
7023,2026-05-15,Introduced in House,House,H,2026-07-17T20:36:05Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:10Z,00,119.0,102,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-19,https://api.congress.gov/v3/bill/119/hconres/1...,R000621,https://api.congress.gov/v3/bill/119/hconres/1...
7024,2026-05-20,Introduced in House,House,H,2026-07-18T16:29:53Z,<p>This concurrent resolution directs the Pres...,2026-07-18T16:29:58Z,00,119.0,103,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-09-22,https://api.congress.gov/v3/bill/119/hconres/1...,A000380,https://api.congress.gov/v3/bill/119/hconres/1...
7025,2026-05-20,Introduced in House,House,H,2026-07-17T20:36:47Z,<p>This concurrent resolution directs the Pres...,2026-07-17T20:36:52Z,00,119.0,104,House,H,"Directing the President, pursuant to section 5...",HCONRES,2026-07-17,https://api.congress.gov/v3/bill/119/hconres/1...,B001313,https://api.congress.gov/v3/bill/119/hconres/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26737,2025-02-25,Introduced in Senate,Senate,S,2026-09-23T16:14:06Z,<p>This resolution designates February 23-Marc...,2026-09-23T16:14:34Z,00,119.0,95,Senate,S,A resolution expressing support for the design...,SRES,2026-09-23,https://api.congress.gov/v3/bill/119/sres/95?f...,S001217,https://api.congress.gov/v3/bill/119/sres/95?f...
26738,2025-02-25,Introduced in Senate,Senate,S,2025-05-07T19:53:39Z,<p>This resolution designates February 24-Febr...,2025-05-07T19:54:03Z,00,119.0,96,Senate,S,A resolution designating the week of February ...,SRES,2025-07-01,https://api.congress.gov/v3/bill/119/sres/96?f...,C001035,https://api.congress.gov/v3/bill/119/sres/96?f...
26740,2025-02-26,Introduced in Senate,Senate,S,2025-05-08T19:34:05Z,<p>This resolution condemns China's Hong Kong ...,2025-05-08T19:34:25Z,00,119.0,98,Senate,S,A resolution condemning Beijing's destruction ...,SRES,2025-05-08,https://api.congress.gov/v3/bill/119/sres/98?f...,R000584,https://api.congress.gov/v3/bill/119/sres/98?f...
26741,2025-02-26,Introduced in Senate,Senate,S,2025-09-19T20:56:40Z,<p>This resolution recognizes Black History Mo...,2025-09-19T20:57:04Z,00,119.0,99,Senate,S,A resolution celebrating Black History Month.,SRES,2026-03-03,https://api.congress.gov/v3/bill/119/sres/99?f...,B001288,https://api.congress.gov/v3/bill/119/sres/99?f...


In [22]:
bills = bills.drop('url', axis=1)
bills.to_csv("../data/processed/bills.csv", index=False)

In [27]:
print(pandas_df_to_dbml(bills, 'bills'))

Table bills {
  actiondate varchar
  actiondesc varchar
  currentchamber varchar
  currentchambercode varchar
  lastsummaryupdatedate varchar
  text varchar
  updatedate varchar
  versioncode varchar
  bill_congress float
  bill_number varchar
  bill_originchamber varchar
  bill_originchambercode varchar
  bill_title varchar
  bill_type varchar
  bill_updatedateincludingtext varchar
  bill_url varchar
  bioguide_id varchar
}



In [ ]:
pd.set_option('display.max_rows', 128)
schedule_e_2026.head(3).T